# Re-download Daily Prices — All 992 RICs

The original download lost most stocks due to Refinitiv gateway timeouts (batch size 40, 1s sleep).
This notebook fixes that:
- **Batch size 10** (was 40)
- **3s sleep** between batches
- **3 retries** with exponential backoff
- **Chunk-level checkpoints** — skip chunks already saved to CSV
- **Optional diagnostic probe** — see how many RICs actually have data before full download

Start date: **2008**. Chunks: 2008–2009, 2010–2011, 2012–2014, 2015–2017, 2018–2020, 2021–2023, 2024–2025.

In [24]:
import refinitiv.data as rd
import pandas as pd
import numpy as np
from datetime import datetime
import time
import json
import os
import warnings
warnings.filterwarnings('ignore')

rd.open_session()

<refinitiv.data.session.Definition object at 0x11ee58530 {name='workspace'}>

In [25]:
# ── Config ─────────────────────────────────────────────────────────────
DATA_DIR      = '../data'
DAILY_BATCH   = 10
SLEEP_BETWEEN = 3
MAX_RETRIES   = 3

CHUNKS = [
    (datetime(2008, 1, 1),  datetime(2009, 12, 31)),
    (datetime(2010, 1, 1),  datetime(2011, 12, 31)),
    (datetime(2012, 1, 1),  datetime(2014, 12, 31)),
    (datetime(2015, 1, 1),  datetime(2017, 12, 31)),
    (datetime(2018, 1, 1),  datetime(2020, 12, 31)),
    (datetime(2021, 1, 1),  datetime(2023, 12, 31)),
    (datetime(2024, 1, 1),  datetime(2025, 12, 31)),
]

all_rics = pd.read_csv(f'{DATA_DIR}/idx_ric_list.csv')['RIC'].tolist()

# Default: download all RICs. If you ran the probe and want to filter,
# set USE_PROBE_FILTER = True below.
download_rics = all_rics

print(f'RICs to download: {len(download_rics)}')
print(f'Batches per chunk: {(len(download_rics) + DAILY_BATCH - 1) // DAILY_BATCH}')
print(f'Total batches: {len(CHUNKS) * ((len(download_rics) + DAILY_BATCH - 1) // DAILY_BATCH)}')

RICs to download: 992
Batches per chunk: 100
Total batches: 700


In [26]:
def fetch_batch(rics, start, end, retries=MAX_RETRIES):
    """Download daily prices for a batch of RICs with retries."""
    for attempt in range(1, retries + 1):
        try:
            df = rd.get_history(
                universe=rics,
                fields=['TR.PriceClose', 'TR.Volume'],
                interval='1D',
                start=start.strftime('%Y-%m-%d'),
                end=end.strftime('%Y-%m-%d')
            )
            if df is not None and not df.empty:
                return df, None
            else:
                return None, None
        except Exception as e:
            err = str(e)[:100]
            if attempt < retries:
                wait = SLEEP_BETWEEN * (2 ** attempt)
                time.sleep(wait)
            else:
                return None, err
    return None, 'max retries exceeded'

## (Optional) Diagnostic probe

Fetches 1 year of data for all 992 RICs to see how many actually have price data.
Takes ~10 min. **Skip this cell if you just want to download everything.**

In [27]:
# ── SKIP THIS CELL to download all 992 RICs without filtering ─────────
probe_start = datetime(2024, 1, 1)
probe_end   = datetime(2024, 12, 31)

probe_has_price = set()
probe_no_data   = set()
probe_failed    = []
bpc = (len(all_rics) + DAILY_BATCH - 1) // DAILY_BATCH

print(f'Probing {len(all_rics)} RICs for 2024 data...\n')

for i in range(0, len(all_rics), DAILY_BATCH):
    batch = all_rics[i:i + DAILY_BATCH]
    batch_num = i // DAILY_BATCH + 1
    
    df, err = fetch_batch(batch, probe_start, probe_end, retries=2)
    
    if df is not None:
        stacked = df.stack(level=0).reset_index()
        stacked.columns = ['Date', 'Instrument', 'Price', 'Volume']
        stacked['Price'] = pd.to_numeric(stacked['Price'], errors='coerce')
        has_data = set(stacked.dropna(subset=['Price'])['Instrument'].unique())
        probe_has_price.update(has_data)
        probe_no_data.update(set(batch) - has_data)
    elif err:
        probe_failed.extend(batch)
    else:
        probe_no_data.update(batch)
    
    if batch_num % 20 == 0:
        print(f'  {batch_num}/{bpc} — has data: {len(probe_has_price)}, '
              f'no data: {len(probe_no_data)}, failed: {len(probe_failed)}')
    time.sleep(1)

print(f'\n{"="*55}')
print(f'  DIAGNOSTIC RESULTS (2024 probe)')
print(f'{"="*55}')
print(f'RICs with price data:  {len(probe_has_price)}')
print(f'RICs with NO data:     {len(probe_no_data)}')
print(f'RICs that failed:      {len(probe_failed)}')

# ── Optionally filter download list ──
USE_PROBE_FILTER = True

if USE_PROBE_FILTER and probe_has_price:
    download_rics = sorted(probe_has_price)
    print(f'\nFiltered to {len(download_rics)} RICs with data '
          f'(skipping {len(all_rics) - len(download_rics)} dead RICs)')
else:
    download_rics = all_rics
    print(f'\nUsing full list: {len(download_rics)} RICs')

Probing 992 RICs for 2024 data...

  20/100 — has data: 196, no data: 4, failed: 0
  40/100 — has data: 386, no data: 14, failed: 0
  60/100 — has data: 580, no data: 20, failed: 0
  80/100 — has data: 772, no data: 28, failed: 0
  100/100 — has data: 958, no data: 34, failed: 0

  DIAGNOSTIC RESULTS (2024 probe)
RICs with price data:  958
RICs with NO data:     34
RICs that failed:      0

Filtered to 958 RICs with data (skipping 34 dead RICs)


## Download all chunks

Each chunk saves to `daily_chunk_N.csv`. If a chunk file already exists, it is skipped.

In [28]:
for chunk_idx, (chunk_start, chunk_end) in enumerate(CHUNKS):
    chunk_file = f'{DATA_DIR}/daily_chunk_{chunk_idx}.csv'
    
    if os.path.exists(chunk_file):
        existing = pd.read_csv(chunk_file)
        print(f'\nChunk {chunk_idx} ({chunk_start.date()} to {chunk_end.date()}): '
              f'ALREADY DONE — {len(existing):,} rows, '
              f'{existing["Instrument"].nunique()} stocks')
        continue
    
    print(f'\n{"="*65}')
    print(f'  Chunk {chunk_idx}: {chunk_start.date()} to {chunk_end.date()}')
    print(f'{"="*65}')
    
    frames = []
    permanently_failed = []
    total_batches = (len(download_rics) + DAILY_BATCH - 1) // DAILY_BATCH
    
    for i in range(0, len(download_rics), DAILY_BATCH):
        batch_rics = download_rics[i:i + DAILY_BATCH]
        batch_num = i // DAILY_BATCH + 1
        
        df_wide, err = fetch_batch(batch_rics, chunk_start, chunk_end)
        
        if df_wide is not None:
            # Convert to long format immediately — do NOT concat wide frames
            # vertically; that creates duplicate date indices and date-based
            # dedup would silently discard all but the first batch.
            df_clean = df_wide.dropna(how='all')
            df_long = df_clean.stack(level=0).reset_index()
            df_long.columns = ['Date', 'Instrument', 'Price', 'Volume']
            df_long[['Price', 'Volume']] = df_long[['Price', 'Volume']].apply(
                pd.to_numeric, errors='coerce'
            )
            frames.append(df_long)
        elif err:
            permanently_failed.append({
                'rics': batch_rics, 'error': err
            })
        
        if batch_num % 10 == 0:
            print(f'  Batch {batch_num}/{total_batches} — '
                  f'{len(frames)} successful, {len(permanently_failed)} failed')
        
        time.sleep(SLEEP_BETWEEN)
    
    if frames:
        # frames are already in long format — safe to concat vertically
        chunk_long = pd.concat(frames, ignore_index=True)
        chunk_long = chunk_long.drop_duplicates(subset=['Date', 'Instrument'], keep='first')
        chunk_long = chunk_long.sort_values(['Instrument', 'Date']).reset_index(drop=True)
        chunk_long.to_csv(chunk_file, index=False)
        print(f'\n  Saved {chunk_file}: {len(chunk_long):,} rows, '
              f'{chunk_long["Instrument"].nunique()} stocks')
    else:
        print(f'\n  No data for chunk {chunk_idx}!')
    
    if permanently_failed:
        print(f'  Permanently failed: {len(permanently_failed)} batches')
        fail_file = f'{DATA_DIR}/daily_chunk_{chunk_idx}_failed.json'
        with open(fail_file, 'w') as f:
            json.dump(permanently_failed, f, indent=2)
        print(f'  Failed RICs saved to {fail_file}')

print(f'\n{"="*65}')
print('  ALL CHUNKS COMPLETE')
print(f'{"="*65}')



  Chunk 0: 2008-01-01 to 2009-12-31
  Batch 10/96 — 10 successful, 0 failed
  Batch 20/96 — 20 successful, 0 failed
  Batch 30/96 — 29 successful, 0 failed
  Batch 40/96 — 39 successful, 0 failed
  Batch 50/96 — 49 successful, 0 failed
  Batch 60/96 — 59 successful, 0 failed
  Batch 70/96 — 69 successful, 0 failed
  Batch 80/96 — 79 successful, 0 failed
  Batch 90/96 — 88 successful, 0 failed

  Saved ../data/daily_chunk_0.csv: 195,425 rows, 405 stocks

  Chunk 1: 2010-01-01 to 2011-12-31
  Batch 10/96 — 10 successful, 0 failed
  Batch 20/96 — 20 successful, 0 failed
  Batch 30/96 — 30 successful, 0 failed
  Batch 40/96 — 40 successful, 0 failed
  Batch 50/96 — 50 successful, 0 failed
  Batch 60/96 — 60 successful, 0 failed
  Batch 70/96 — 70 successful, 0 failed
  Batch 80/96 — 80 successful, 0 failed
  Batch 90/96 — 90 successful, 0 failed

  Saved ../data/daily_chunk_1.csv: 223,543 rows, 454 stocks

  Chunk 2: 2012-01-01 to 2014-12-31
  Batch 10/96 — 10 successful, 0 failed
  Batch

An error occurred while requesting URL('http://localhost:9005/api/udf').
	ReadError('[Errno 54] Connection reset by peer')


  Batch 50/96 — 50 successful, 0 failed
  Batch 60/96 — 60 successful, 0 failed
  Batch 70/96 — 70 successful, 0 failed
  Batch 80/96 — 80 successful, 0 failed
  Batch 90/96 — 90 successful, 0 failed

  Saved ../data/daily_chunk_2.csv: 389,650 rows, 532 stocks

  Chunk 3: 2015-01-01 to 2017-12-31
  Batch 10/96 — 10 successful, 0 failed
  Batch 20/96 — 20 successful, 0 failed
  Batch 30/96 — 30 successful, 0 failed
  Batch 40/96 — 40 successful, 0 failed
  Batch 50/96 — 50 successful, 0 failed
  Batch 60/96 — 60 successful, 0 failed
  Batch 70/96 — 70 successful, 0 failed
  Batch 80/96 — 80 successful, 0 failed
  Batch 90/96 — 90 successful, 0 failed

  Saved ../data/daily_chunk_3.csv: 437,572 rows, 600 stocks

  Chunk 4: 2018-01-01 to 2020-12-31
  Batch 10/96 — 10 successful, 0 failed
  Batch 20/96 — 20 successful, 0 failed
  Batch 30/96 — 30 successful, 0 failed
  Batch 40/96 — 40 successful, 0 failed
  Batch 50/96 — 50 successful, 0 failed
  Batch 60/96 — 60 successful, 0 failed
  Ba

## Combine all chunks

In [29]:
all_chunks = []
for chunk_idx in range(len(CHUNKS)):
    chunk_file = f'{DATA_DIR}/daily_chunk_{chunk_idx}.csv'
    if os.path.exists(chunk_file):
        df = pd.read_csv(chunk_file, parse_dates=['Date'])
        all_chunks.append(df)
        print(f'  Loaded {chunk_file}: {len(df):,} rows, {df["Instrument"].nunique()} stocks')
    else:
        print(f'  MISSING: {chunk_file}')


daily_long = pd.concat(all_chunks, ignore_index=True)
daily_long = daily_long.drop_duplicates(subset=['Date', 'Instrument'], keep='first')
daily_long = daily_long.sort_values(['Instrument', 'Date']).reset_index(drop=True)

print(f'\nCombined daily panel: {len(daily_long):,} rows, '
      f'{daily_long["Instrument"].nunique()} stocks')
print(f'Date range: {daily_long["Date"].min().date()} to {daily_long["Date"].max().date()}')
print(f'Price coverage: {daily_long["Price"].notna().mean()*100:.1f}%')

daily_long.to_csv(f'{DATA_DIR}/idx_daily_prices.csv', index=False)
print('\nSaved idx_daily_prices.csv')

  Loaded ../data/daily_chunk_0.csv: 195,425 rows, 405 stocks
  Loaded ../data/daily_chunk_1.csv: 223,543 rows, 454 stocks
  Loaded ../data/daily_chunk_2.csv: 389,650 rows, 532 stocks
  Loaded ../data/daily_chunk_3.csv: 437,572 rows, 600 stocks
  Loaded ../data/daily_chunk_4.csv: 555,608 rows, 763 stocks
  Loaded ../data/daily_chunk_5.csv: 673,567 rows, 920 stocks
  Loaded ../data/daily_chunk_6.csv: 453,276 rows, 958 stocks

Combined daily panel: 2,927,717 rows, 958 stocks
Date range: 2001-10-26 to 2025-12-30
Price coverage: 70.1%

Saved idx_daily_prices.csv


## Rebuild master panel

In [30]:
jci = pd.read_csv(f'{DATA_DIR}/jci_index.csv', parse_dates=['Date'])
rf  = pd.read_csv(f'{DATA_DIR}/bi_riskfree_rate.csv', parse_dates=['Date'])

master = pd.merge(daily_long, jci[['Date', 'Market_Return']], on='Date', how='left')
master = pd.merge(master, rf[['Date', 'Daily_Rf']], on='Date', how='left')

master = master.sort_values(['Instrument', 'Date'])
master['Stock_Return'] = master.groupby('Instrument')['Price'].transform(
    lambda x: np.log(x / x.shift(1))
)
master['Excess_Return'] = master['Stock_Return'] - master['Daily_Rf']
master['Market_Excess'] = master['Market_Return'] - master['Daily_Rf']
master['DayOfWeek'] = master['Date'].dt.dayofweek
master['DayName']   = master['Date'].dt.day_name()
master['YearMonth'] = master['Date'].dt.to_period('M')

master.to_csv(f'{DATA_DIR}/idx_master_panel.csv', index=False)

valid = master.dropna(subset=['Stock_Return'])
print(f'Master panel: {len(master):,} rows, {master["Instrument"].nunique()} stocks')
print(f'Valid stock returns: {len(valid):,} ({len(valid)/len(master)*100:.1f}%)')
print(f'\nSaved idx_master_panel.csv')

Master panel: 2,927,717 rows, 958 stocks
Valid stock returns: 1,957,446 (66.9%)

Saved idx_master_panel.csv


## Quality check

In [31]:
valid = master.dropna(subset=['Stock_Return']).copy()
valid['Year'] = valid['Date'].dt.year
print('Stocks with valid returns by year:')
print(valid.groupby('Year')['Instrument'].nunique().to_string())

pre  = valid[(valid['Date'] >= '2008-01-01') & (valid['Date'] <= '2016-12-31')]
post = valid[(valid['Date'] >= '2017-01-01') & (valid['Date'] <= '2024-12-31')]
print(f'\nPre  (2008-2016): {len(pre):,} obs, {pre["Instrument"].nunique()} stocks')
print(f'Post (2017-2024): {len(post):,} obs, {post["Instrument"].nunique()} stocks')

both = set(pre['Instrument'].unique()) & set(post['Instrument'].unique())
print(f'In both periods: {len(both)} stocks')

Stocks with valid returns by year:
Year
2008    348
2009    349
2010    383
2011    405
2012    432
2013    462
2014    479
2015    498
2016    509
2017    543
2018    599
2019    652
2020    694
2021    719
2022    757
2023    822
2024    853
2025    833

Pre  (2008-2016): 619,463 obs, 553 stocks
Post (2017-2024): 1,152,765 obs, 915 stocks
In both periods: 520 stocks


## Retry failures

In [32]:
import glob as globmod

fail_files = sorted(globmod.glob(f'{DATA_DIR}/daily_chunk_*_failed.json'))
if not fail_files:
    print('No failed batches to retry!')
else:
    for ff in fail_files:
        chunk_idx = int(os.path.basename(ff).split('_')[2])
        chunk_start, chunk_end = CHUNKS[chunk_idx]
        
        with open(ff) as f:
            failures = json.load(f)
        
        print(f'\nRetrying {len(failures)} failed batches from chunk {chunk_idx}...')
        recovered = []
        still_failed = []
        
        for fail in failures:
            df_wide, err = fetch_batch(fail['rics'], chunk_start, chunk_end, retries=5)
            if df_wide is not None:
                df_clean = df_wide.dropna(how='all')
                df_long = df_clean.stack(level=0).reset_index()
                df_long.columns = ['Date', 'Instrument', 'Price', 'Volume']
                df_long[['Price', 'Volume']] = df_long[['Price', 'Volume']].apply(
                    pd.to_numeric, errors='coerce'
                )
                recovered.append(df_long)
                print(f'  Recovered: {fail["rics"][:3]}...')
            elif err:
                still_failed.append(fail)
            time.sleep(SLEEP_BETWEEN * 2)
        
        if recovered:
            rec_long = pd.concat(recovered, ignore_index=True)
            chunk_file = f'{DATA_DIR}/daily_chunk_{chunk_idx}.csv'
            existing = pd.read_csv(chunk_file, parse_dates=['Date'])
            combined = pd.concat([existing, rec_long], ignore_index=True)
            combined = combined.drop_duplicates(subset=['Date', 'Instrument'], keep='first')
            combined.to_csv(chunk_file, index=False)
            print(f'  +{len(rec_long):,} rows recovered')
        
        if still_failed:
            with open(ff, 'w') as f:
                json.dump(still_failed, f, indent=2)
            print(f'  {len(still_failed)} still failed')
        else:
            os.remove(ff)
            print(f'  All recovered!')
    
    print('\nRe-run Combine + Rebuild cells above.')


No failed batches to retry!
